In [17]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)
DEEPSEEK_API_KEY= os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL= os.getenv("DEEPSEEK_BASE_URL")
llmOpenai = init_chat_model(
    model="gpt-5.4-mini",
    # api_key=DEEPSEEK_API_KEY,
    # base_url=DEEPSEEK_BASE_URL,
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
)

class WeatherSchema(BaseModel):
    city: str = Field(default="北京", description="城市名称")
    if_forecast: bool = Field(default=False, description="是否包含明日天气预"
                                                         "报")

@tool(args_schema=WeatherSchema, description="获取指定城市的天气信息。")
def get_weather(city: str,if_forecast : bool):
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res

molde_with_tools = llmOpenai.bind_tools([get_weather])

messages = [
    HumanMessage("武汉明天和后天天气怎么样")
]

tool_calls = molde_with_tools.invoke(messages).tool_calls
for tool in tool_calls:
    if tool["name"] == "get_weather":
        response = get_weather.invoke(tool["args"])
        messages.append(response)

response = llmOpenai.invoke(messages)
messages.append(response)
print(response)

武汉 今天天气不错
明天也不错

content='如果按你说的，**武汉今天天气不错，明天也不错**。  \n但我这边**无法直接获取实时天气预报**，所以**后天天气**我不能准确确认。\n\n如果你愿意，我可以帮你：\n1. **整理成一句更自然的天气描述**\n2. **根据你提供的温度/天气情况，帮你判断后天大概会怎样**\n3. **教你怎么快速查武汉明后天天气**' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 106, 'prompt_tokens': 28, 'total_tokens': 134, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DyYclDuTgJSs3bw8NlyrN2UcMZH2y', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f3676-2fe1-70e2-9da2-4aa0be420bba-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 106, 'total_tokens': 134, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reason

In [18]:
from langchain_core.messages import HumanMessage, ToolMessage
from pydantic import BaseModel, Field
from langchain_core.tools import tool

@tool(parse_docstring=True)
def get_weather(city: str,if_forecast : bool):
    """
    查询当日天气，可以包含明日天气预报

    Args:
        city: 城市名称
        if_forecast: 是否包含明日天气预报
    """
    res = f"{city} 今天天气不错"
    if if_forecast:
        res += "\n明天也不错"
    return res

molde_with_tools = llmOpenai.bind_tools([get_weather])

messages = [
    HumanMessage("武汉明天和后天天气怎么样")
]

tool_calls = molde_with_tools.invoke(messages).tool_calls
for tool in tool_calls:
    if tool["name"] == "get_weather":
        response = get_weather.invoke(tool["args"])
        messages.append(response)

response = llmOpenai.invoke(messages)
messages.append(response)
print(response)

content='看起来你已经知道一些情况了：**武汉今天不错，明天也不错**。  \n但我这边**不能直接获取实时天气**，所以没法准确确认“后天”或给出最新预报。\n\n如果你愿意，我可以帮你：\n1. **根据你提供的天气截图/文字**，整理成更清楚的三天天气；\n2. **教你快速查看武汉明后天天气**的方法；\n3. 你把天气预报截图发来，我帮你解读是否适合出门、穿什么。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 28, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-DyYeluPFg1ydfR5KZfARWPIoEtXuP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019f3678-153e-7dc1-879e-d519bcdf01ca-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 28, 'output_tokens': 124, 'total_tokens': 152, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

## 多工具调用

In [44]:
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool
from rich import print as rprint
# 1.定义工具
# 定义股票查询工具
@tool(parse_docstring=True)
def get_stock_price(company: str, timeframe: str = "today") -> str:
    """获取指定公司的股票价格信息

    Args:
        company: 公司名称（如：苹果公司, 微软公司, 谷歌公司）
        timeframe: 时间范围（today-今日, week-本周, month-本月）
    """
    # 模拟股票数据
    mock_data = {
        "苹果公司": {"today": 185.20, "week": 183.50, "month": 180.75},
        "微软公司": {"today": 415.86, "week": 412.30, "month": 405.42},
        "谷歌公司": {"today": 15.42, "week": 15.20, "month": 14.85}
    }
    if company in mock_data:
        price = mock_data[company].get(timeframe, "未知时间范围")
        return f"{company} {timeframe}价格: {price}美元"
    else:
        return f"未找到股票代码 {company} 的数据"


# 定义新闻搜索工具
@tool(parse_docstring=True)
def search_news(company: str) -> str:
    """搜索指定公司的财经新闻

    Args:
        company: 公司名称

    Returns:公司的财经新闻，每个新闻占一行
    """
    # 模拟新闻数据
    mock_news = {
        "苹果公司": [
            "苹果发布新款iPhone，股价上涨3%",
            "苹果与欧盟达成反垄断和解协议",
            "苹果将在印度扩大生产规模"
        ],
        "微软公司": [
            "微软Azure云业务季度增长超预期",
            "微软完成对Nuance的收购",
            "微软推出新一代AI助手Copilot"
        ],
        "谷歌公司": [
            "谷歌发布新AI模型，性能提升20%",
            "谷歌与OpenAI合作，开发新的AI助手",
            "谷歌在欧洲展开AI研究项目"
        ]
    }
    news_list = mock_news.get(company, [f"未找到{company}的相关新闻"])
    return "\n".join(news_list)

# 2.初始化模型并绑定工具
tools = [get_stock_price, search_news]
model_with_tools = llmOpenai.bind_tools(tools)
message_list = []
human_message = HumanMessage(content="苹果公司今天的股价是多少？最近有什么新闻？")
# human_message = HumanMessage(content="比较一下微软和苹果的股价")
# human_message = HumanMessage(content="腾讯最近有什么重大新闻？")
# human_message = HumanMessage(content="海水为什么是咸的？")
message_list.append(human_message)

while True:
    response = model_with_tools.invoke(message_list)
    message_list.append(response)

    if not response.tool_calls:
        break

    response.pretty_print()
    for tool_call in response.tool_calls:
        if tool_call["name"] == "get_stock_price":
            result = get_stock_price.invoke(tool_call["args"])
            # ✅ 包装成 ToolMessage
            message_list.append(ToolMessage(
                content=result,
                tool_call_id=tool_call["id"]
            ))
        if tool_call["name"] == "search_news":
            news_result = search_news.invoke(tool_call["args"])
            # ✅ 包装成 ToolMessage
            message_list.append(ToolMessage(
                content=news_result,
                tool_call_id=tool_call["id"]
            ))
for msg in message_list:
    msg.pretty_print()

================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_ZrADu7c98IpKU2iIpDtY2kCR)
 Call ID: call_ZrADu7c98IpKU2iIpDtY2kCR
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_2EHUwFPhns7u1iv8HrVmlVfS)
 Call ID: call_2EHUwFPhns7u1iv8HrVmlVfS
  Args:
    company: 苹果公司
================================ Human Message =================================

苹果公司今天的股价是多少？最近有什么新闻？
================================== Ai Message ==================================
Tool Calls:
  get_stock_price (call_ZrADu7c98IpKU2iIpDtY2kCR)
 Call ID: call_ZrADu7c98IpKU2iIpDtY2kCR
  Args:
    company: 苹果公司
    timeframe: today
  search_news (call_2EHUwFPhns7u1iv8HrVmlVfS)
 Call ID: call_2EHUwFPhns7u1iv8HrVmlVfS
  Args:
    company: 苹果公司
================================= Tool Message =================================

苹果公司 today价格: 185.2美元
================================= Tool Message =================================

苹果发布新款iPhone，股价上涨3%
苹果与欧盟达